# Representations, methods & extremes

The defaults give a good aggregation, but three knobs let you control *what* the
typical periods preserve: the **clustering method**, the **representation**, and
whether to force **extreme periods**.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## Clustering method

How periods are grouped. `hierarchical` (the default) and `kmeans` are good
general choices; `kmedoids`/`kmaxoids` pick real periods as centers; `averaging`
is the simplest. Compare the per-column error:

In [ ]:
from tsam import ClusterConfig

methods = ["hierarchical", "kmeans", "kmedoids", "kmaxoids", "averaging"]
errors = {}
for m in methods:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=m)
    )
    errors[m] = r.accuracy.rmse
pd.DataFrame(errors).round(4)

## Representation

How each cluster becomes one profile. `mean` smooths; `medoid` (default) picks a
real day; `distribution` reproduces the value distribution (great for storage
models); `minmax_mean` keeps the extremes per time step. The **duration curve**
shows the difference best — it ignores timing and checks the spread of values:

In [ ]:
reps = ["medoid", "mean", "distribution", "minmax_mean"]
frames = []
for rep in reps:
    r = tsam.aggregate(
        data,
        n_clusters=6,
        period_duration="1D",
        cluster=ClusterConfig(representation=rep),
    )
    sorted_load = (
        r.reconstructed["Load"].sort_values(ascending=False).reset_index(drop=True)
    )
    frames.append(
        pd.DataFrame(
            {
                "rank": range(len(sorted_load)),
                "Load": sorted_load.values,
                "representation": rep,
            }
        )
    )
orig = data["Load"].sort_values(ascending=False).reset_index(drop=True)
frames.append(
    pd.DataFrame(
        {"rank": range(len(orig)), "Load": orig.values, "representation": "original"}
    )
)
px.line(
    pd.concat(frames),
    x="rank",
    y="Load",
    color="representation",
    title="Duration curve by representation",
)

## Force extreme periods

Clustering finds *typical* behaviour, so the single peak day can get averaged
away. [`ExtremeConfig`](../api/configuration.md) forces chosen extremes to be kept
exactly. Without it the peak is clipped; with it, recovered:

In [ ]:
from tsam import ExtremeConfig

base = tsam.aggregate(data, n_clusters=6, period_duration="1D")
kept = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    extremes=ExtremeConfig(method="new_cluster", max_value=["Load"]),
)
print(f"original peak Load:       {data['Load'].max():.1f}")
print(f"default typical peak:     {base.cluster_representatives['Load'].max():.1f}")
print(f"with extreme period kept: {kept.cluster_representatives['Load'].max():.1f}")

Extreme days enter as their own clusters, so they have a small occurrence count:

In [ ]:
kept.plot.cluster_counts()